# 🧪 LAB: Manual MLPs for Classification and Regression

In this lab, you will use `PyTorch` to implement manually a multi-layer perceptron (MLP) for three different tasks: binary classification, multi-class classification and regression. 

## General instructions to complete in ALL three tasks:

1. ***IMPLEMENTATION***:
   
Implement a separate class for each task:

  - `BinaryMLP` for the binary classification task  
  - `MultiClassMLP` for the multi-class classification task  
  - `RegressionMLP` for the regression task

   Each class must include the following methods:

  - `__init__` for initializing 1 or 2 hidden layers. (*Help*: having a second hidden layer only means the need of having a extra set of weights and intercept). 
  - `forward` to transfer information from the input to the output layer. (*Help*: make sure that information passes through a second hidden layer if there is one).
  - `cost` computing the cost.
  - `fit` for training, using the gradients to manually update the parameters of the model. **Use stochastic gradient descent to update your weights**. *N.B.* You may probably reuse much of the code we used in this week tutorial already. 
  - `predict` to convert the information at the output layer into the required output.

2. ***DATA PREPARATION***

For each task, you will be provided with a toy dataset. For each dataset:

  - Split into training and holdout sets (use an 80/20 split)
  - Standardize the features properly, avoiding data leakage
  - Convert all data into `PyTorch` Tensors for compatibility

3. ***MODEL TRAINING AND EVALUATION***

Instantiate the model for each task and train it under different hyperparameter configurations. In each case, record the performance on **both** the training and holdout sets using the appropriate metric for the task (accuracy, MSE, etc.). You should explore the following configurations:

  - One hidden layer, varying the number of hidden units (use ReLU as their activation function)
  - Same number of hidden units across one or two hidden layers (use ReLU as their activation function)
  - Repeat the above setups using Tanh activation instead of ReLU

Present your results in a compact way (e.g., a summary table, a data frame etc).

**NOTE**: When training your model, use a fixed learning rate of your choice (e.g., 0.01 is a reasonable starting point) and a reasonably large number of  epochs (e.g., 100–200) based on how training and holdout performance evolve.

4. ***REFLECTION AND DISCUSSION***

Reflect on the impact of the different hyperparameter settings:

- How does the number of hidden units affect performance?
- What changes when using two layers instead of one?
- How does the activation function (ReLU vs. Tanh) influence results?

Please elaborate your answers.

---

**Collaboration Note**: This assignment is designed to support collaborative work. We encourage you to divide tasks among group members so that everyone can contribute meaningfully. Many components of the assignment can be approached in parallel or split logically across team members. Good coordination and thoughtful integration of your work will lead to a stronger final result.

**Ideally, each group member should be responsible for one of the separate tasks.** BUT, everyone should help each other along the way, both reviewing and refining results and discussion.

---

In total, this lab assignment will be worth **100 points**.

--- 
**Submission notes**:

* Write down all group members' names, or at least the group name (if you have one and you previously provided it), in the first cell of the notebook.

* Verify that the notebook runs as expected and that all required outputs are included.


In [4]:
# NAME(s) = "Izra(Jamal) Abudiab, Mustafa Ali, Rashi Adhikari"

## 1. Pre-implementation Group Discussion (15 points)

Discuss and agree on:

- What cost function should be used for each of the below task.
- What changes are needed in the output layer for each of these tasks. In particular, consider the number of units and the activation function.
- Why it is important to standardize the data before training each model.
- How you could detect overfitting when training your models.

1. Binary Classification could use binary cross entropy. For the Regression MLP it could use Mean Squared Error for evaluation. For the multi-class MLP, it would use Categorical Cross-Entropy.

2. Sigmoid would be an appropriate activation function for the binary classification and it would output one unit which is the probability of the two cases. For the multi-class MLP, we would use Softmax and the numbers returned would be the probability for each of the individual cases which will end up summing to one. The number of units for multi-class MLP would be as many classes that are in the model. We do not need any activiation function for regression MLP, and the output would be just one number representing our expected continuous value. 

3. Without standardization of the data, the MLP's will be more unstable which is to say: 
- They will converge on average slower
- They will be dominated by certain features based on units of features
- It can exacerbate the vanishing gradient problem by pushing neurons in one direction or the other more strongly without uncertainty


4. We could detect overfitting by comparing our testing accuracy with our training accuracy since as our training accuracy begins to have diminishing returns and our testing accuracy is actually declining.

## 2. Binary Classification (25 points)

Use the dataset below to complete points 1 to 4 in the general instructions for this task.

Use as many cells as needed.

In [14]:
from sklearn.datasets import make_moons

X, y = make_moons(n_samples=1000, noise=0.2, random_state=42)

In [15]:
#1 

import torch

class BinaryMLP:
    def __init__(self, input_size=2, hidden_size=8, num_layers=1, activation="relu"):
        self.num_layers = num_layers
        self.activation = activation

        self.W1 = (torch.randn(input_size, hidden_size) * 0.1).requires_grad_()
        self.b1 = torch.zeros(hidden_size, requires_grad=True)
        if num_layers == 2:
            self.W2 = (torch.randn(hidden_size, hidden_size) * 0.1).requires_grad_()
            self.b2 = torch.zeros(hidden_size, requires_grad=True)
        self.W_out = (torch.randn(hidden_size, 1) * 0.1).requires_grad_()
        self.b_out = torch.zeros(1, requires_grad=True)

    def forward(self, X):
        h = X @ self.W1 + self.b1
        h = torch.relu(h) if self.activation == "relu" else torch.tanh(h)

        if self.num_layers == 2:
            h = h @ self.W2 + self.b2
            h = torch.relu(h) if self.activation == "relu" else torch.tanh(h)
        return torch.sigmoid(h @ self.W_out + self.b_out)

    def cost(self, y_pred, y):
        y = y.reshape(-1, 1).float()
        y_pred = torch.clamp(y_pred, 1e-7, 1 - 1e-7)
        return -(y * torch.log(y_pred) +
                 (1 - y) * torch.log(1 - y_pred)).mean()


    def fit(self, X, y, epochs=100, lr=0.01):
        y = y.reshape(-1, 1).float()
        params = [self.W1, self.b1, self.W_out, self.b_out]
        if self.num_layers == 2:
            params += [self.W2, self.b2]
        for epoch in range(epochs):
            indices = torch.randperm(X.shape[0])
            for i in indices:
                y_pred = self.forward(X[i:i+1])
                loss = self.cost(y_pred, y[i:i+1])
                gradients = torch.autograd.grad(loss, params)
                with torch.no_grad():
                    for p, grad in zip(params, gradients):
                        p -= lr * grad

        return self

    def predict(self, X):
        with torch.no_grad():
            return (self.forward(X) >= 0.5).int()


In [16]:
#2 

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X_train, X_holdout, y_train, y_holdout = train_test_split(
    X, y, test_size=0.2, random_state=42
)
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_holdout = scaler.transform(X_holdout)

X_train = torch.tensor(X_train, dtype=torch.float32)
X_holdout = torch.tensor(X_holdout, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32)
y_holdout = torch.tensor(y_holdout, dtype=torch.float32)


In [17]:
#3 

import pandas as pd
learning_rate = 0.01
epochs = 100
results = []

for activation in ["relu", "tanh"]:
    for num_layers in [1, 2]:
        for hidden_size in [4, 8]:
            torch.manual_seed(42)

            model = BinaryMLP(
                input_size=2,
                hidden_size=hidden_size,
                num_layers=num_layers,
                activation=activation
            )
            model.fit(X_train, y_train, epochs=epochs, lr=learning_rate)
            train_pred = model.predict(X_train).reshape(-1)
            holdout_pred = model.predict(X_holdout).reshape(-1)
            train_accuracy = (train_pred == y_train).float().mean().item()
            holdout_accuracy = (holdout_pred == y_holdout).float().mean().item()

            results.append({
                "Activation": activation,
                "Hidden Layers": num_layers,
                "Hidden Units": hidden_size,
                "Train Accuracy": train_accuracy,
                "Holdout Accuracy": holdout_accuracy
            })

results_df = pd.DataFrame(results)
results_df



,Activation,Hidden Layers,Hidden Units,Train Accuracy,Holdout Accuracy
0,relu,1,4,0.86500,0.850
1,relu,1,8,0.97250,0.985
2,relu,2,4,0.86750,0.850
3,relu,2,8,0.96250,0.970
4,tanh,1,4,0.86500,0.855
5,tanh,1,8,0.96000,0.990
6,tanh,2,4,0.86500,0.850
7,tanh,2,8,0.96125,0.980


### 4 

1. 
Increasing the number of hidden units from 4 to 8 improved the model's performance. To elaborate, one hidden layer and ReLU activation increased training accuracy from 86.5% to 97.25%, while the holdout accuracy increased from 85% to 98.5%. This shows that having more hidden units allowed the model to capture the nonlinear patterns in the data. With only 4 units, the model may have had difficulty learning the curved decision boundary. However, adding more hidden units may also increase the chance of overfitting. 

2. 
Adding the second hidden layer did not consistently improve performance. For example, with 8 hidden units and ReLU activation, holdout accuracy decreased from 98.5% with one layer to 97% with two layers. This also occurred with Tanh, where the holdout accuracy decreased from 99% to 98%- a small but negative effect. This suggests that one hidden layer was enough to capture the patterns in the dataset. Adding a second layer increased the model complexity but did not necessarily improve accuracy for the simpler dataset. 

3. 
ReLU and Tanh produced similar results overall, although Tanh performed slightly better under some configurations. The highest holdout accuracy was 99% using Tanh, one hidden layer, and 8 hidden units, as compared to the 98.5% accuracy with ReLU. Also, ReLU outputs zero for negative inputs and keeps positive inputs, while Tanh transforms values into a range between -1 and 1. This affects how the models learn nonlinear relationships. Overall, because the differences between the two models were small, hidden units seem to have a larger impact on performance than ReLU and Tanh. 

## 2. Multi-Class Classification (25 points)

Use the dataset below to complete points 1 to 4 in the general instructions for this task.

Use as many cells as needed.

---
**NOTE**: This will likely be the most challenging exercise. To help you, here are some pointers:

 - The **output (last) layer** should have as many units as there are classes in your data.  

 - The **activation function of the output layer must be softmax**, not sigmoid. Softmax ensures that all output values are between 0 and 1 and sum to 1, so they can be interpreted as probabilities across the different classes. (*Help*: you may want to look into `torch.nn.functional` to find its implementation...)

 - For a multi-class problem, your target variable `y` should be **one-hot encoded**. For example:  
   - Label = 0 → [1, 0, 0]  
   - Label = 1 → [0, 1, 0]  
   - Label = 2 → [0, 0, 1]  
   You can easily achieve this with `OneHotEncoder` from `sklearn.preprocessing`.  

 - The **predicted class** corresponds to the unit with the highest probability.  
   Example:  
   - `[0.1, 0.3, 0.6] → class 2`  
   - `[0.6, 0.2, 0.2] → class 0`  

 - For this exercise, you need to **implement the categorical cross-entropy loss** (an extension of binary cross-entropy to multiple classes). It is defined as:  

   $$\sum_{c=1}^{l} y_{o,c}\,\log(p_{o,c}),$$

   where $l$ is the number of classes, $y_{o,c}$ is the one-hot encoded label for observation $o$, and $p_{o,c}$ is the predicted probability for class $c$ (after applying softmax). The log is the natural logarithm.

---

In [18]:
from sklearn.datasets import make_classification

X, y = make_classification(n_samples=1000, n_classes=3, 
                           n_clusters_per_class=1, 
                           n_features=2, n_informative=2, n_redundant=0, random_state=1234, flip_y=0.15)

In [22]:
# 1
import torch
import torch.nn.functional as F
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# 80/20 split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1234)

# standardize using only training stats (avoid leakage)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# one-hot encode the labels
encoder = OneHotEncoder(sparse_output=False)
y_train_oh = encoder.fit_transform(y_train.reshape(-1, 1))
y_test_oh = encoder.transform(y_test.reshape(-1, 1))

# convert everything to tensors
X_train_t = torch.tensor(X_train, dtype=torch.float32)
X_test_t = torch.tensor(X_test, dtype=torch.float32)
y_train_t = torch.tensor(y_train_oh, dtype=torch.float32)
y_test_t = torch.tensor(y_test_oh, dtype=torch.float32)

In [24]:
#2 

class MultiClassMLP:
    def __init__(self, n_features, n_classes, n_hidden1, n_hidden2=None, activation='relu'):
        self.n_hidden2 = n_hidden2
        self.activation = activation

        self.W1 = (torch.randn(n_features, n_hidden1) * 0.1).requires_grad_()
        self.b1 = torch.zeros(n_hidden1, requires_grad=True)

        if n_hidden2 is not None:
            self.W2 = (torch.randn(n_hidden1, n_hidden2) * 0.1).requires_grad_()
            self.b2 = torch.zeros(n_hidden2, requires_grad=True)
            out_in = n_hidden2
        else:
            out_in = n_hidden1

        self.Wout = (torch.randn(out_in, n_classes) * 0.1).requires_grad_()
        self.bout = torch.zeros(n_classes, requires_grad=True)

    def _activate(self, z):
        if self.activation == 'relu':
            return torch.relu(z)
        elif self.activation == 'tanh':
            return torch.tanh(z)

    def forward(self, X):
        h1 = self._activate(X @ self.W1 + self.b1)
        if self.n_hidden2 is not None:
            h2 = self._activate(h1 @ self.W2 + self.b2)
            logits = h2 @ self.Wout + self.bout
        else:
            logits = h1 @ self.Wout + self.bout
        return F.softmax(logits, dim=1)

    def cost(self, y_pred, y_true):
        eps = 1e-8
        return -torch.mean(torch.sum(y_true * torch.log(y_pred + eps), dim=1))

    def params(self):
        p = [self.W1, self.b1, self.Wout, self.bout]
        if self.n_hidden2 is not None:
            p += [self.W2, self.b2]
        return p

    def fit(self, X_train, y_train, lr=0.01, epochs=150):
        losses = []
        for epoch in range(epochs):
            y_pred = self.forward(X_train)
            loss = self.cost(y_pred, y_train)
            loss.backward()
            with torch.no_grad():
                for p in self.params():
                    p -= lr * p.grad
                    p.grad.zero_()
            losses.append(loss.item())
        return losses

    def predict(self, X):
        with torch.no_grad():
            probs = self.forward(X)
            return torch.argmax(probs, dim=1)

In [29]:
#3
def accuracy(preds, y_true_t):
    true_labels = torch.argmax(y_true_t, dim=1)
    return (preds == true_labels).float().mean().item()

results = []

# --- Setup 1: one hidden layer, varying hidden units (ReLU) ---
for units in [4, 8, 16, 32]:
    model = MultiClassMLP(n_features=X_train_t.shape[1], n_classes=3,
                           n_hidden1=units, n_hidden2=None, activation='relu')
    model.fit(X_train_t, y_train_t, lr=0.01, epochs=150)
    train_acc = accuracy(model.predict(X_train_t), y_train_t)
    test_acc = accuracy(model.predict(X_test_t), y_test_t)
    results.append({
        'setup': '1 layer, varying units (ReLU)',
        'hidden_layers': 1, 'hidden_units': units, 'activation': 'relu',
        'train_acc': train_acc, 'test_acc': test_acc
    })

# --- Setup 2: same hidden units across 1 vs 2 hidden layers (ReLU) ---
for units in [8, 16]:
    for n_layers in [1, 2]:
        n_hidden2 = units if n_layers == 2 else None
        model = MultiClassMLP(n_features=X_train_t.shape[1], n_classes=3,
                               n_hidden1=units, n_hidden2=n_hidden2, activation='relu')
        model.fit(X_train_t, y_train_t, lr=0.01, epochs=150)
        train_acc = accuracy(model.predict(X_train_t), y_train_t)
        test_acc = accuracy(model.predict(X_test_t), y_test_t)
        results.append({
            'setup': '1 vs 2 layers, same units (ReLU)',
            'hidden_layers': n_layers, 'hidden_units': units, 'activation': 'relu',
            'train_acc': train_acc, 'test_acc': test_acc
        })

# --- Setup 3: repeat Setup 1, but with Tanh instead of ReLU ---
for units in [4, 8, 16, 32]:
    model = MultiClassMLP(n_features=X_train_t.shape[1], n_classes=3,
                           n_hidden1=units, n_hidden2=None, activation='tanh')
    model.fit(X_train_t, y_train_t, lr=0.01, epochs=150)
    train_acc = accuracy(model.predict(X_train_t), y_train_t)
    test_acc = accuracy(model.predict(X_test_t), y_test_t)
    results.append({
        'setup': '1 layer, varying units (Tanh)',
        'hidden_layers': 1, 'hidden_units': units, 'activation': 'tanh',
        'train_acc': train_acc, 'test_acc': test_acc
    })

# --- Setup 4: repeat Setup 2, but with Tanh instead of ReLU ---
for units in [8, 16]:
    for n_layers in [1, 2]:
        n_hidden2 = units if n_layers == 2 else None
        model = MultiClassMLP(n_features=X_train_t.shape[1], n_classes=3,
                               n_hidden1=units, n_hidden2=n_hidden2, activation='tanh')
        model.fit(X_train_t, y_train_t, lr=0.01, epochs=150)
        train_acc = accuracy(model.predict(X_train_t), y_train_t)
        test_acc = accuracy(model.predict(X_test_t), y_test_t)
        results.append({
            'setup': '1 vs 2 layers, same units (Tanh)',
            'hidden_layers': n_layers, 'hidden_units': units, 'activation': 'tanh',
            'train_acc': train_acc, 'test_acc': test_acc
        })

In [30]:
#4
results_df = pd.DataFrame(results)[['setup', 'hidden_layers', 'hidden_units', 'activation', 'train_acc', 'test_acc']]
results_df

,setup,hidden_layers,hidden_units,activation,train_acc,test_acc
0,"1 layer, varying units (ReLU)",1,4,relu,0.59125,0.590
1,"1 layer, varying units (ReLU)",1,8,relu,0.72000,0.765
2,"1 layer, varying units (ReLU)",1,16,relu,0.70625,0.725
3,"1 layer, varying units (ReLU)",1,32,relu,0.75375,0.775
4,"1 vs 2 layers, same units (ReLU)",1,8,relu,0.69875,0.700
5,"1 vs 2 layers, same units (ReLU)",2,8,relu,0.33750,0.315
6,"1 vs 2 layers, same units (ReLU)",1,16,relu,0.73625,0.765
7,"1 vs 2 layers, same units (ReLU)",2,16,relu,0.57875,0.595
8,"1 layer, varying units (Tanh)",1,4,tanh,0.62875,0.630
9,"1 layer, varying units (Tanh)",1,8,tanh,0.60625,0.610


Increasing the number of hidden units generally improved test accuracy for a single hidden layer, rising from 0.59 with 4 units to 0.775 with 32 units for ReLU, and similarly from 0.63 to 0.755 for Tanh, though the gains weren't perfectly monotonic (16 units occasionally underperformed 8). Moving from one hidden layer to two had a much bigger, and more unpredictable, effect than we expected. With ReLU, the 2-layer network's test accuracy dropped sharply compared to its 1-layer counterpart, in one case falling to 0.315, close to random guessing for three classes. This looks like a dead ReLU problem: with our small weight initialization, enough units in the second hidden layer likely got pushed into the flat, zero-gradient part of ReLU early in training and never recovered. Tanh didn't show this collapse, its 2-layer results stayed closer to its 1-layer results and even improved slightly at 16 units, suggesting Tanh's bounded, non-zero gradients made the deeper network easier to train with manual SGD. Overall, ReLU and Tanh performed similarly for single-layer networks, but Tanh was noticeably more stable once we added a second hidden layer.

## 3: Regression Task (25 points)

Use the dataset below to complete points 1 to 4 in the general instructions for this task.

Use as many cells as needed.

In [21]:
from sklearn.datasets import make_regression
X, y = make_regression(n_samples=1000, n_features=2, n_informative=2, random_state=1234, noise=75)

In [22]:
# USE AS MANY CELLS AS NEEDED

## 4. Discussion (5 points)

You created a separate class for each task and likely repeated much of the same code across implementations.

Discuss within your group how could you have leveraged inheritance to make your code more reusable and avoid duplication. Provide examples. Be specific.

YOUR TEXT HERE

## 5. Collaboration Reflection (5 points)

As a group, identify **one specific challenge or inefficiency** you encountered while working on this lab. In 3–5 sentences:

1. Briefly describe what happened and how it affected your work.
2. Identify one concrete action your group will take to avoid or better manage this issue in the next lab.

If your group encountered no major problem, identify one aspect of your collaboration that could still be improved.

For this lab, each of us took full ownership of one task, Rashi on binary classification, Mustafa on multi-class, and Izra on regression, which let us work in parallel without stepping on each other’s code. The main inefficiency was that we didn’t agree ahead of time on a shared structure for the MLP classes (how helper methods like params() or _activate() should be named and organized), so when we compared our implementations there were small inconsistencies that took extra time to reconcile when discussing the inheritance question. Next time, we’ll agree on a common class skeleton or naming convention before splitting up the tasks, so the final notebook is more consistent and easier to merge.